# Free Cooking Video Assembler (Android + Google Colab)

Create a 60-second vertical cooking montage from video clips you upload. This notebook uses FFmpeg in the Colab runtime; it does **not** call paid AI video-generation APIs.

## Before you start
- Open this notebook in Google Colab on your Android phone.
- Upload only footage you own or have permission to use. The notebook is not preloaded with anyone's videos.
- You can optionally upload `voiceover.mp3` and/or `music.mp3` with your clips. Music is mixed quietly under narration.
- If you upload only a couple of short clips, the notebook may repeat them to reach 60 seconds.
- Colab's free runtime has limits and may disconnect. Download your finished MP4 when it is ready.

**Output:** 1080 × 1920, 9:16 portrait, 30 fps, MP4 (H.264).

In [ ]:
# Step 1 — Install/check FFmpeg and upload your media
import os, shutil, subprocess, glob, json, math
from google.colab import files

if shutil.which('ffmpeg') is None:
    subprocess.run(['apt-get','-qq','update'], check=True)
    subprocess.run(['apt-get','-qq','install','-y','ffmpeg'], check=True)

WORK = '/content/cooking_video'
os.makedirs(WORK, exist_ok=True)
print('Select your cooking video clips. You may also select voiceover.mp3 and music.mp3 in the same upload.')
uploaded = files.upload()
for name, data in uploaded.items():
    safe_name = os.path.basename(name).replace('/', '_')
    with open(os.path.join(WORK, safe_name), 'wb') as f:
        f.write(data)

video_exts = ('.mp4','.mov','.m4v','.webm','.mkv')
video_files = sorted([os.path.join(WORK, n) for n in os.listdir(WORK) if n.lower().endswith(video_exts)])
voice_files = [os.path.join(WORK,n) for n in os.listdir(WORK) if n.lower().startswith('voiceover.')]
music_files = [os.path.join(WORK,n) for n in os.listdir(WORK) if n.lower().startswith('music.')]
print('\nVideo clips found:', len(video_files))
for p in video_files: print(' •', os.path.basename(p))
print('Voiceover:', bool(voice_files), '| Background music:', bool(music_files))
if not video_files:
    raise RuntimeError('No video files found. Upload at least one MP4/MOV/WebM clip and rerun this cell.')

## Step 2 — Build the 60-second montage

The default uses about 8 seconds from each clip, then cycles through the uploaded clips until the timeline reaches 60 seconds. Change `SEGMENT_SECONDS` if you want faster or slower cuts. The video is center-cropped to portrait; check the result because important action near the edges may be cropped.

In [ ]:
# Step 2 — Render the vertical montage
TOTAL_SECONDS = 60
SEGMENT_SECONDS = 8
WIDTH, HEIGHT, FPS = 1080, 1920, 30
segments = math.ceil(TOTAL_SECONDS / SEGMENT_SECONDS)
chosen = [video_files[i % len(video_files)] for i in range(segments)]
durations = [min(SEGMENT_SECONDS, TOTAL_SECONDS - i*SEGMENT_SECONDS) for i in range(segments)]

cmd = ['ffmpeg','-y']
for p in chosen:
    cmd += ['-stream_loop','-1','-i',p]
filters = []
for i, dur in enumerate(durations):
    filters.append(f'[{i}:v]scale={WIDTH}:{HEIGHT}:force_original_aspect_ratio=increase,crop={WIDTH}:{HEIGHT},fps={FPS},setsar=1,trim=duration={dur},setpts=PTS-STARTPTS[v{i}]')
concat_inputs = ''.join(f'[v{i}]' for i in range(len(durations)))
filters.append(f'{concat_inputs}concat=n={len(durations)}:v=1:a=0[outv]')
cmd += ['-filter_complex',';'.join(filters),'-map','[outv]','-an','-t',str(TOTAL_SECONDS),'-c:v','libx264','-preset','veryfast','-crf','20','-pix_fmt','yuv420p','-movflags','+faststart','/content/cooking_video_silent.mp4']
print('Rendering', len(chosen), 'segments…')
result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
if result.returncode != 0:
    print(result.stdout[-5000:])
    raise RuntimeError('FFmpeg video rendering failed. See the error output above.')
print('Silent video created: /content/cooking_video_silent.mp4')

## Step 3 — Add optional voiceover and music

If you uploaded `voiceover.mp3`, it is used as the main audio. If you also uploaded `music.mp3`, the music is mixed quietly underneath. If you uploaded only music, it becomes the soundtrack. If you uploaded neither, the export remains silent.

In [ ]:
# Step 3 — Mix audio if supplied, then export the final MP4
base = '/content/cooking_video_silent.mp4'
final_path = '/content/creamy_garlic_parmesan_pasta_60s.mp4'
voice = voice_files[0] if voice_files else None
music = music_files[0] if music_files else None

if voice and music:
    audio_cmd = ['ffmpeg','-y','-i',base,'-i',voice,'-stream_loop','-1','-i',music,
      '-filter_complex',f'[1:a]apad,atrim=0:{TOTAL_SECONDS}[v];[2:a]volume=0.14,atrim=0:{TOTAL_SECONDS}[m];[v][m]amix=inputs=2:duration=longest:dropout_transition=2[a]',
      '-map','0:v:0','-map','[a]','-t',str(TOTAL_SECONDS),'-c:v','copy','-c:a','aac','-b:a','192k','-movflags','+faststart',final_path]
elif voice:
    audio_cmd = ['ffmpeg','-y','-i',base,'-i',voice,'-filter_complex',f'[1:a]apad,atrim=0:{TOTAL_SECONDS}[a]', '-map','0:v:0','-map','[a]','-t',str(TOTAL_SECONDS),'-c:v','copy','-c:a','aac','-b:a','192k','-movflags','+faststart',final_path]
elif music:
    audio_cmd = ['ffmpeg','-y','-i',base,'-stream_loop','-1','-i',music,'-map','0:v:0','-map','1:a:0','-t',str(TOTAL_SECONDS),'-c:v','copy','-c:a','aac','-b:a','192k','-af','volume=0.25','-movflags','+faststart',final_path]
else:
    shutil.copyfile(base, final_path)

if voice or music:
    result = subprocess.run(audio_cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if result.returncode != 0:
        print(result.stdout[-5000:])
        raise RuntimeError('Audio mixing failed. You can still download the silent video from /content/cooking_video_silent.mp4.')

print('Finished:', final_path)
print('File size (MB):', round(os.path.getsize(final_path)/1024/1024, 1))
files.download(final_path)

## Step 4 — Review before publishing

Watch the exported video from beginning to end on your phone. Check that the important cooking actions are visible, the clip order makes sense, the audio is not too loud, and you have permission to use every source clip and music track. If the crop hides the pan or food, use a source clip with the subject centered or adjust the scale/crop settings in Step 2.

### Important limitation
This notebook edits and combines footage; it does not create new AI-generated cooking scenes, write narration, or automatically generate subtitles. Those can be added as separate steps using free tools, subject to their current limits.